# ECERC Limitation 2, Experiment 8a: zero-shot LLM soft labels for the training split

Experiment 7 improved ECERC by fusing it with a frozen zero-shot LLM **at inference**. Experiment 8 asks whether a single ECERC model, trained with that LLM's soft labels, keeps part of the gain **without** the LLM. This first notebook only produces the soft labels. It runs the same LLM as Experiment 7 (Qwen2.5-32B-Instruct, 4-bit, same revision, prompt and scoring) on every utterance of the **training** dialogues.

- No label of any split is given to the LLM. The test split is not scored and its labels are not read.
- Before the long pass, the first 20 dev dialogues are scored again and compared with Experiment 7's saved output, to confirm the model is the same.
- The output `train_llm.npz` goes into the second notebook (8b), which trains and evaluates ECERC.

Expect about 7.5 hours on Kaggle **GPU T4 x2** (the model needs both GPUs) with Internet **On**. Run it with **Save Version → Save & Run All (Commit)**.

In [ ]:
# ===== Configuration: leave as is. (Set TRUST_DATA_PICKLE = True only if the data cell asks you to.) =====
import os

TRUST_DATA_PICKLE = False

if os.path.isdir('/kaggle/working'):
    WORK_DIR = '/kaggle/working/ecerc_l2'
elif os.path.isdir('/content'):
    WORK_DIR = '/content/ecerc_l2'
else:
    WORK_DIR = os.path.abspath('ecerc_l2_work')
CONFIG = dict(
    work_dir=WORK_DIR,
    cache_dir='/tmp/ecerc_l2_cache',        # the 19 GB model download; not part of the saved output
    data_file_id='1IQokygWgjaZuSXvCsfIcUzj1Gu69hDCb',            # authors' Google Drive (linked from the official ECERC README)
    data_filename='meld_emotion_semantic_features_roberta.pkl',
    expected_data_sha256='a0349614b2b58e8b2db768f0144aa36b96fb2348ca436a6879ee2623cc2d2f80',  # observed in run_colab_1
    trust_data_pickle=TRUST_DATA_PICKLE,
    expected_split_utterances=dict(train=9989, valid=1109, test=2610),
    expected_split_dialogues=dict(train=[1038, 1039], valid=114, test=280),
    llm_model='unsloth/Qwen2.5-32B-Instruct-bnb-4bit',  # 4-bit copy of Qwen/Qwen2.5-32B-Instruct, as in Experiment 7
    llm_revision='aa79e3472818bdec779075d80928602591d9f2a0',
    llm_max_memory='13GiB',                 # per GPU; the 4-bit weights are about 19 GB in total
    llm_min_gpu_gb=28,                      # two T4s; one 16 GB GPU is not enough
    expected_label_token_ids=[[20628, 58694, 59568, 86907], [12761, 19784, 23043, 84484], [69, 8679, 41967, 90371], [30681, 50878, 59665, 82114], [4123, 15888, 27138, 79771], [4093, 4243, 4839, 67062], [4003, 7416, 10743, 19234]],
    expected_prompt_sha256='903ff9c018bfd4db07d250a97cb50c880192827344206f760d680e3a7dfa7016',
    reference_dev_dump='embedded',          # Experiment 7's dev log-probabilities (dev_llm.npz, embedded below)
    repro_dialogues=20, repro_min_agreement=0.95,
    offline=False,                          # True only for the local smoke test
)
os.makedirs(WORK_DIR, exist_ok=True)
print('Working directory:', WORK_DIR)

## 1. Code
`l2_posthoc.py`, `l2_pathb.py`, `l2_expert.py` and `l2_zeroshot.py` from the earlier experiments (unchanged), `l2_distill_teacher.py`, and Experiment 7's dev log-probabilities, embedded as base64 and checked by SHA-256. Readable copies are in `research/limitation2/`.

In [ ]:
# Writes the analysis code and Experiment 7's dev log-probabilities, checks their SHA-256, then sets up.
import base64, hashlib, importlib, sys

CODE_DIR = os.path.join(CONFIG['work_dir'], 'code')
os.makedirs(CODE_DIR, exist_ok=True)
CODE_FILES = {  # name: (sha256, base64 content)
    'l2_posthoc.py': ('...
fSBNQiknKQogICAgcmV0dXJuIHpwYXRoCg==
'''),
    'l2_pathb.py': ('...
fSBNQiknKQogICAgcmV0dXJuIHpwYXRoCg==
'''),
    'l2_expert.py': ('...
TUIpJykKICAgIHJldHVybiB6cGF0aAo=
'''),
    'l2_zeroshot.py': ('...
'''),
    'l2_distill_teacher.py': ('861cb12493ac2111598e5e56bde47a6ca3d5aba56cd0b560f9de0db54ae3984c', '''
IiIiCkVDRVJDIExpbWl0YXRpb24gMiwgRXhwZXJpbWVudCA4YTogemVyby1zaG90IExMTSBzb2Z0
IGxhYmVscyBmb3IgdGhlIHRyYWluaW5nIHNwbGl0LgoKTmV3IGNvZGUgZm9yIHJlc2VhcmNoL2xp
bWl0YXRpb24yLiBFeHBlcmltZW50IDcgZnVzZWQgRUNFUkMgd2l0aCBhIGZyb3plbiB6ZXJvLXNo
b3QgTExNIGF0IGluZmVyZW5jZS4gRXhwZXJpbWVudCA4CmFza3Mgd2hldGhlciBhIHNpbmdsZSBF
Q0VSQyBtb2RlbCB0cmFpbmVkIHdpdGggdGhhdCBMTE0ncyBzb2Z0IGxhYmVscyBrZWVwcyBwYXJ0
IG9mIHRoZSBnYWluIHdpdGggbm8gTExNIGF0IGluZmVyZW5jZS4KVGhpcyBmaXJzdCBwYXJ0IHJ1
bnMgdGhlIExMTSBvZiBFeHBlcmltZW50IDcsIHVuY2hhbmdlZCAobW9kZWwgcmV2aXNpb24sIHBy
b21wdCwgbGFiZWwgdG9rZW5zLCBjYWNoZWQgc2NvcmluZyksIG9uIHRoZQp1dHRlcmFuY2VzIG9m
IHRoZSBUUkFJTklORyBkaWFsb2d1ZXMgYW5kIHNhdmVzIGl0cyBsb2ctcHJvYmFiaWxpdGllcyBm
b3IgdGhlIGRpc3RpbGxhdGlvbiBub3RlYm9vayAoOGIpLgoKTGFiZWwtZnJlZTogdGhlIExMTSBy
ZWFkcyBvbmx5IHRoZSB0ZXh0IGFuZCBzcGVha2Vycy4gVGhlIHRlc3Qgc3BsaXQgaXMgbm90IHNj
b3JlZCBhbmQgbm8gdGVzdCBsYWJlbCBpcyByZWFkLiBCZWZvcmUKdGhlIHRyYWluaW5nIHNwbGl0
IGlzIHNjb3JlZCwgdGhlIGZpcnN0IGRldiBkaWFsb2d1ZXMgYXJlIHNjb3JlZCBhZ2FpbiBhbmQg
Y29tcGFyZWQgd2l0aCBFeHBlcmltZW50IDcncyBzYXZlZCBkZXYKbG9nLXByb2JhYmlsaXRpZXMs
IHNvIHRoZSB0ZWFjaGVyIGlzIHRoZSBzYW1lIG1vZGVsIGFzIGluIEV4cGVyaW1lbnQgNy4KIiIi
CmltcG9ydCBpbXBvcnRsaWIudXRpbAppbXBvcnQgb3MKaW1wb3J0IHBsYXRmb3JtCmltcG9ydCBz
dWJwcm9jZXNzCmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKaW1wb3J0IHppcGZpbGUKCmltcG9ydCBu
dW1weSBhcyBucAppbXBvcnQgdG9yY2gKCmltcG9ydCBsMl9wb3N0aG9jIGFzIFAgICAjIGRhdGEg
Z2F0ZSwgbG9nZ2luZyBhbmQgY2hlY2tzICh1bmNoYW5nZWQpCmltcG9ydCBsMl9wYXRoYiBhcyBM
QiAgICAjIGRhdGEgZG93bmxvYWQgKHVuY2hhbmdlZCkKaW1wb3J0IGwyX3plcm9zaG90IGFzIExa
ICAjIEV4cGVyaW1lbnQgNzogcHJvbXB0LCBsYWJlbCB0b2tlbnMsIG1vZGVsIGxvYWRpbmcgYW5k
IGNhY2hlZCBzY29yaW5nICh1bmNoYW5nZWQpCgpDLCBDTEFTU0VTID0gUC5DLCBQLkNMQVNTRVMK
CgpkZWYgc3RhZ2Vfc2V0dXAoQ09ORklHLCBjb2RlX2Rpcik6CiAgICBvdXQgPSBvcy5wYXRoLmpv
aW4oQ09ORklHWyd3b3JrX2RpciddLCAnb3V0cHV0c19kaXN0aWxsX3RlYWNoZXInKQogICAgZm9y
IHN1YiBpbiAoJycsICdkdW1wcycpOgogICAgICAgIG9zLm1ha2VkaXJzKG9zLnBhdGguam9pbihv
dXQsIHN1YiksIGV4aXN0X29rPVRydWUpCiAgICBvcy5tYWtlZGlycyhvcy5wYXRoLmpvaW4oQ09O
RklHWydjYWNoZV9kaXInXSwgJ2hmJyksIGV4aXN0X29rPVRydWUpCiAgICBjdWRhID0gdG9yY2gu
Y3VkYS5pc19hdmFpbGFibGUoKQogICAgaWYgY3VkYTogICMgNC1iaXQgbG9hZGluZyBuZWVkcyBi
aXRzYW5kYnl0ZXMgYW5kIGFjY2VsZXJhdGUgKGFzIGluIEV4cGVyaW1lbnQgNykKICAgICAgICBu
ZWVkID0gW3AgZm9yIHAgaW4gKCdiaXRzYW5kYnl0ZXMnLCAnYWNjZWxlcmF0ZScpIGlmIGltcG9y
dGxpYi51dGlsLmZpbmRfc3BlYyhwKSBpcyBOb25lXQogICAgICAgIGlmIG5lZWQ6CiAgICAgICAg
ICAgIHN1YnByb2Nlc3MucnVuKFtzeXMuZXhlY3V0YWJsZSwgJy1tJywgJ3BpcCcsICdpbnN0YWxs
JywgJy1xJ10gKyBuZWVkLCBjaGVjaz1GYWxzZSkKICAgIGltcG9ydCB0cmFuc2Zvcm1lcnMKICAg
IGlmIENPTkZJRy5nZXQoJ3JlZmVyZW5jZV9kZXZfZHVtcCcpID09ICdlbWJlZGRlZCc6ICAjIEV4
cGVyaW1lbnQgNydzIGRldl9sbG0ubnB6LCB3cml0dGVuIGJ5IHRoZSBjb2RlIGNlbGwKICAgICAg
ICBDT05GSUdbJ3JlZmVyZW5jZV9kZXZfZHVtcCddID0gb3MucGF0aC5qb2luKGNvZGVfZGlyLCAn
ZGV2X2xsbS5ucHonKQogICAgUyA9IGRpY3QoY2ZnPUNPTkZJRywgb3V0PW91dCwgY29kZV9kaXI9
Y29kZV9kaXIsIGNoZWNrcz1bXSwgdG9yY2g9dG9yY2gpCiAgICBTWydkZXZpY2UnXSA9ICdjdWRh
JyBpZiBjdWRhIGVsc2UgJ2NwdScKICAgIGlmIGNvZGVfZGlyIG5vdCBpbiBzeXMucGF0aDoKICAg
ICAgICBzeXMucGF0aC5pbnNlcnQoMCwgY29kZV9kaXIpCiAgICB2ZXJzaW9ucyA9IHtwOiBfX2lt
cG9ydF9fKHApLl9fdmVyc2lvbl9fIGlmIGltcG9ydGxpYi51dGlsLmZpbmRfc3BlYyhwKSBlbHNl
IE5vbmUKICAgICAgICAgICAgICAgIGZvciBwIGluICgnYml0c2FuZGJ5dGVzJywgJ2FjY2VsZXJh
dGUnKX0KICAgIGdwdXMgPSBbKHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKGkpLCByb3VuZCh0
b3JjaC5jdWRhLmdldF9kZXZpY2VfcHJvcGVydGllcyhpKS50b3RhbF9tZW1vcnkgLyAyICoqIDMw
LCAxKSkKICAgICAgICAgICAgZm9yIGkgaW4gcmFuZ2UodG9yY2guY3VkYS5kZXZpY2VfY291bnQo
KSldIGlmIGN1ZGEgZWxzZSBbXQogICAgU1snZW52J10gPSBkaWN0KHB5dGhvbj1wbGF0Zm9ybS5w
eXRob25fdmVyc2lvbigpLCBwbGF0Zm9ybT1wbGF0Zm9ybS5wbGF0Zm9ybSgpLCB0b3JjaD10b3Jj
aC5fX3ZlcnNpb25fXywKICAgICAgICAgICAgICAgICAgICB0cmFuc2Zvcm1lcnM9dHJhbnNmb3Jt
ZXJzLl9fdmVyc2lvbl9fLCBudW1weT1ucC5fX3ZlcnNpb25fXywgZGV2aWNlPVNbJ2RldmljZSdd
LCBncHVzPWdwdXMsCiAgICAgICAgICAgICAgICAgICAgKip2ZXJzaW9ucywgY29kZV9zaGEyNTY9
e2Y6IFAuc2hhMjU2X2ZpbGUob3MucGF0aC5qb2luKGNvZGVfZGlyLCBmKSkKICAgICAgICAgICAg
ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZm9yIGYgaW4gc29ydGVkKG9zLmxpc3Rk
aXIoY29kZV9kaXIpKSBpZiBmLmVuZHN3aXRoKCcucHknKX0pCiAgICBQLndyaXRlX2pzb24ob3Mu
cGF0aC5qb2luKG91dCwgJ2Vudmlyb25tZW50Lmpzb24nKSwgU1snZW52J10pCiAgICBQLndyaXRl
X2pzb24ob3MucGF0aC5qb2luKG91dCwgJ2Rlc2lnbi5qc29uJyksICAjIHdyaXR0ZW4gYmVmb3Jl
IGFueSBkYXRhIGlzIGxvYWRlZAogICAgICAgICAgICAgICAgIGRpY3QobGxtPWRpY3QobW9kZWw9
Q09ORklHWydsbG1fbW9kZWwnXSwgcmV2aXNpb249Q09ORklHLmdldCgnbGxtX3JldmlzaW9uJykp
LAogICAgICAgICAgICAgICAgICAgICAgcHJvbXB0PWRpY3Qoc3lzdGVtPUxaLlNZU1RFTSwgcXVl
c3Rpb249TFouUVVFU1RJT04sIHNoYTI1Nj1MWi5QUk9NUFRfU0hBMjU2KSwKICAgICAgICAgICAg
ICAgICAgICAgIHNjb3Jpbmc9IkV4cGVyaW1lbnQgNydzLCB1bmNoYW5nZWQgKGwyX3plcm9zaG90
LmxsbV9sb2dwcm9icykiLCBzcGxpdF9zY29yZWQ9J3RyYWluJywKICAgICAgICAgICAgICAgICAg
ICAgIHJlcHJvZHVjaWJpbGl0eT1kaWN0KGRldl9kaWFsb2d1ZXM9Q09ORklHWydyZXByb19kaWFs
b2d1ZXMnXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG1pbl9h
cmdtYXhfYWdyZWVtZW50PUNPTkZJR1sncmVwcm9fbWluX2FncmVlbWVudCddLAogICAgICAgICAg
ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcmVmZXJlbmNlPW9zLnBhdGguYmFzZW5h
bWUoQ09ORklHWydyZWZlcmVuY2VfZGV2X2R1bXAnXSBvciAnbm9uZScpKSwKICAgICAgICAgICAg
ICAgICAgICAgIHRlc3Q9J25vdCBzY29yZWQ7IG5vIHRlc3QgbGFiZWwgaXMgcmVhZCcpKQogICAg
aWYgY3VkYToKICAgICAgICB0b3RhbCA9IHN1bShnWzFdIGZvciBnIGluIGdwdXMpCiAgICAgICAg
UC5yZWNvcmRfY2hlY2soUywgZidHUFUgbWVtb3J5IGlzIGVub3VnaCBmb3IgdGhlIDQtYml0IExM
TSAoPj0ge0NPTkZJR1sibGxtX21pbl9ncHVfZ2IiXX0gR0IgaW4gdG90YWwpJywKICAgICAgICAg
ICAgICAgICAgICAgICB0b3RhbCA+PSBDT05GSUdbJ2xsbV9taW5fZ3B1X2diJ10sIGhhcmQ9VHJ1
ZSwKICAgICAgICAgICAgICAgICAgICAgICBkZXRhaWw9Zid7Z3B1c307IG9uIEthZ2dsZSBjaG9v
c2UgQWNjZWxlcmF0b3IgLT4gR1BVIFQ0IHgyJykKICAgIGVsc2U6CiAgICAgICAgcHJpbnQoJ1dB
Uk5JTkc6IG5vIEdQVS4gQSAzMkIgbW9kZWwgY2Fubm90IHJ1biBvbiBDUFU7IHN3aXRjaCB0aGUg
YWNjZWxlcmF0b3IgdG8gR1BVIFQ0IHgyLicpCiAgICBQLl9zYXZlX2NoZWNrcyhTKQogICAgUC5s
b2dfZXZlbnQoUywgJ3NldHVwJywgZGV2aWNlPVNbJ2RldmljZSddLCBncHVzPWdwdXMsIHRvcmNo
PXRvcmNoLl9fdmVyc2lvbl9fLCB0cmFuc2Zvcm1lcnM9dHJhbnNmb3JtZXJzLl9fdmVyc2lvbl9f
KQogICAgcmV0dXJuIFMKCgpkZWYgc3RhZ2VfcHJlcGFyZShTKToKICAgICIiIk1FTEQgZmVhdHVy
ZXMgKGF1dGhvcnMnIEdvb2dsZSBEcml2ZSkgYW5kIHRoZSBkYXRhIGdhdGU7IHJlYWRzIHRyYWlu
L2RldiBsYWJlbHMgb25seS4iIiIKICAgIExCLnN0YWdlX2Rvd25sb2FkKFMpCiAgICBQLnN0YWdl
X2xvYWRfZGF0YShTKQogICAgRCA9IFNbJ0QnXQogICAgb2sgPSBhbGwobGVuKERbJ3NlbnRlbmNl
cyddW3ZdKSA9PSBsZW4oRFsndmlkZW9BdWRpbyddW3ZdKSBhbmQgYWxsKGlzaW5zdGFuY2UoeCwg
c3RyKSBmb3IgeCBpbiBEWydzZW50ZW5jZXMnXVt2XSkKICAgICAgICAgICAgIGZvciBzIGluICgn
dHJhaW4nLCAndmFsaWQnKSBmb3IgdiBpbiBTWydrZXlzJ11bc10pCiAgICBQLnJlY29yZF9jaGVj
ayhTLCAndXR0ZXJhbmNlIHRleHQgaXMgcHJlc2VudCBmb3IgZXZlcnkgdHJhaW4gYW5kIGRldiB1
dHRlcmFuY2UgKGxhYmVsLWZyZWUgY2hlY2spJywgb2ssIGhhcmQ9VHJ1ZSkKICAgIFAuX3NhdmVf
Y2hlY2tzKFMpCgoKZGVmIHN0YWdlX2xsbShTKToKICAgICIiIkxvYWRzIHRoZSBMTE0sIGNoZWNr
cyBpdCBhZ2FpbnN0IEV4cGVyaW1lbnQgNyBhbmQgc2NvcmVzIHRoZSB0cmFpbmluZyBzcGxpdC4i
IiIKICAgIGNmZyA9IFNbJ2NmZyddCiAgICBtb2RlbCwgdG9rID0gTFoubG9hZF9sbG0oUykKICAg
IFNbJ2xhYmVsX2lkcyddLCB1bmlxdWUgPSBMWi5sYWJlbF90b2tlbl9pZHModG9rKQogICAgc3Bl
bGxlZCA9IHt3OiBbdG9rLmRlY29kZShbaV0pIGZvciBpIGluIGddIGZvciB3LCBnIGluIHppcChM
Wi5MQUJFTFMsIFNbJ2xhYmVsX2lkcyddKX0KICAgIFAucmVjb3JkX2NoZWNrKFMsICdsYWJlbCB3
b3JkcyBzdGFydCB3aXRoIGRpc3RpbmN0IHRva2VucycsIHVuaXF1ZSwgaGFyZD1UcnVlLCBkZXRh
aWw9c3BlbGxlZCkKICAgIGlmIGNmZy5nZXQoJ2V4cGVjdGVkX2xhYmVsX3Rva2VuX2lkcycpIGlz
IG5vdCBOb25lOgogICAgICAgIFAucmVjb3JkX2NoZWNrKFMsICJsYWJlbCB0b2tlbiBpZHMgZXF1
YWwgRXhwZXJpbWVudCA3J3MgZnJvemVuIGlkcyIsCiAgICAgICAgICAgICAgICAgICAgICAgW2xp
c3QoZykgZm9yIGcgaW4gU1snbGFiZWxfaWRzJ11dID09IGNmZ1snZXhwZWN0ZWRfbGFiZWxfdG9r
ZW5faWRzJ10sIGhhcmQ9VHJ1ZSkKICAgIFAucmVjb3JkX2NoZWNrKFMsICJwcm9tcHQgZXF1YWxz
IEV4cGVyaW1lbnQgNydzIChTSEEtMjU2KSIsIExaLlBST01QVF9TSEEyNTYgPT0gY2ZnWydleHBl
Y3RlZF9wcm9tcHRfc2hhMjU2J10sCiAgICAgICAgICAgICAgICAgICBoYXJkPVRydWUsIGRldGFp
bD1MWi5QUk9NUFRfU0hBMjU2WzoxNl0pCgogICAgIyByZXByb2R1Y2liaWxpdHk6IHRoZSBmaXJz
dCBkZXYgZGlhbG9ndWVzIGFnYWluLCBhZ2FpbnN0IEV4cGVyaW1lbnQgNydzIHNhdmVkIGRldiBs
b2ctcHJvYmFiaWxpdGllcwogICAga2V5cyA9IFNbJ2tleXMnXVsndmFsaWQnXVs6Y2ZnWydyZXBy
b19kaWFsb2d1ZXMnXV0KICAgIFNbJ2tleXMnXVsncmVwcm9fZGV2J10gPSBrZXlzCiAgICBscF9k
ZXYsIGluZm9fZGV2ID0gTFoubGxtX2xvZ3Byb2JzKFMsIG1vZGVsLCB0b2ssICdyZXByb19kZXYn
KQogICAgbnAuc2F2ZXpfY29tcHJlc3NlZChvcy5wYXRoLmpvaW4oU1snb3V0J10sICdkdW1wcycs
ICdyZXByb19kZXZfbGxtLm5weicpLCBsb2dwPWxwX2Rldi5hc3R5cGUobnAuZmxvYXQzMiksCiAg
ICAgICAgICAgICAgICAgICAgICAgIGtleXM9bnAuYXNhcnJheShrZXlzKSkKICAgIHJlZl9wYXRo
ID0gY2ZnLmdldCgncmVmZXJlbmNlX2Rldl9kdW1wJykKICAgIHJlcHJvID0gZGljdChkaWFsb2d1
ZXM9bGVuKGtleXMpLCB1dHRlcmFuY2VzPWxlbihscF9kZXYpKQogICAgaWYgcmVmX3BhdGg6CiAg
ICAgICAgcmVmID0gbnAubG9hZChyZWZfcGF0aClbJ2xvZ3AnXVs6bGVuKGxwX2RldildLmFzdHlw
ZShucC5mbG9hdDY0KQogICAgICAgIGFncmVlID0gZmxvYXQoKHJlZi5hcmdtYXgoMSkgPT0gbHBf
ZGV2LmFyZ21heCgxKSkubWVhbigpKQogICAgICAgIHJlcHJvLnVwZGF0ZShyZWZlcmVuY2Vfc2hh
MjU2PVAuc2hhMjU2X2ZpbGUocmVmX3BhdGgpLCBhcmdtYXhfYWdyZWVtZW50PWFncmVlLAogICAg
ICAgICAgICAgICAgICAgICBtYXhfYWJzX2RpZmY9ZmxvYXQobnAuYWJzKHJlZiAtIGxwX2Rldiku
bWF4KCkpLCBtZWFuX2Fic19kaWZmPWZsb2F0KG5wLmFicyhyZWYgLSBscF9kZXYpLm1lYW4oKSkp
CiAgICAgICAgUC5yZWNvcmRfY2hlY2soUywgZiJkZXYgcmUtc2NvcmluZyByZXByb2R1Y2VzIEV4
cGVyaW1lbnQgNyAoYXJnbWF4IGFncmVlbWVudCA+PSB7Y2ZnWydyZXByb19taW5fYWdyZWVtZW50
J119KSIsCiAgICAgICAgICAgICAgICAgICAgICAgYWdyZWUgPj0gY2ZnWydyZXByb19taW5fYWdy
ZWVtZW50J10sIGhhcmQ9VHJ1ZSwKICAgICAgICAgICAgICAgICAgICAgICBkZXRhaWw9Zid7MTAw
ICogYWdyZWU6LjFmfSUgb2Yge2xlbihscF9kZXYpfSB1dHRlcmFuY2VzOyBtYXggfGRpZmZ8IHty
ZXByb1sibWF4X2Fic19kaWZmIl06LjRmfScpCiAgICBlbHNlOgogICAgICAgIFAucmVjb3JkX2No
ZWNrKFMsICdkZXYgcmUtc2NvcmluZyBjb21wYXJlZCB3aXRoIEV4cGVyaW1lbnQgNycsIEZhbHNl
LCBoYXJkPUZhbHNlLCBkZXRhaWw9J25vIHJlZmVyZW5jZSBnaXZlbicpCgogICAgbHAsIGluZm8g
PSBMWi5sbG1fbG9ncHJvYnMoUywgbW9kZWwsIHRvaywgJ3RyYWluJykKICAgIGxlbmd0aHMgPSBu
cC5hcnJheShbbGVuKFNbJ0QnXVsndmlkZW9BdWRpbyddW3ZdKSBmb3IgdiBpbiBTWydrZXlzJ11b
J3RyYWluJ11dKQogICAgUC5yZWNvcmRfY2hlY2soUywgJ29uZSByb3cgcGVyIHRyYWluaW5nIHV0
dGVyYW5jZScsIGxlbihscCkgPT0gaW50KGxlbmd0aHMuc3VtKCkpLCBoYXJkPVRydWUsCiAgICAg
ICAgICAgICAgICAgICBkZXRhaWw9Zid7bGVuKGxwKX0gcm93cycpCiAgICBQLnJlY29yZF9jaGVj
ayhTLCAndGhlIHNldmVuIGxhYmVsIHdvcmRzIGNhcnJ5IG1vc3Qgb2YgdGhlIG5leHQtdG9rZW4g
cHJvYmFiaWxpdHkgb24gdHJhaW4nLAogICAgICAgICAgICAgICAgICAgaW5mb1snbGFiZWxfbWFz
c19tZWFuJ10gPj0gMC41LCBoYXJkPUZhbHNlLCBkZXRhaWw9ZidtZWFuIHtpbmZvWyJsYWJlbF9t
YXNzX21lYW4iXTouM2Z9JykKICAgIHBhdGggPSBvcy5wYXRoLmpvaW4oU1snb3V0J10sICdkdW1w
cycsICd0cmFpbl9sbG0ubnB6JykKICAgIG5wLnNhdmV6X2NvbXByZXNzZWQocGF0aCwgbG9ncD1s
cC5hc3R5cGUobnAuZmxvYXQzMiksIGtleXM9bnAuYXNhcnJheShTWydrZXlzJ11bJ3RyYWluJ10p
LCBsZW5ndGhzPWxlbmd0aHMsCiAgICAgICAgICAgICAgICAgICAgICAgIGRhdGFfc2hhMjU2PW5w
LmFzYXJyYXkoU1snZGF0YV9zaGEyNTYnXSkpCiAgICBTWydsbG1faW5mbyddID0gZGljdChsYWJl
bF90b2tlbnM9c3BlbGxlZCwgcmVwcm9kdWNpYmlsaXR5PXJlcHJvLCByZXByb19kZXY9aW5mb19k
ZXYsIHRyYWluPWluZm8sCiAgICAgICAgICAgICAgICAgICAgICAgICB0cmFpbl9sbG1fbnB6X3No
YTI1Nj1QLnNoYTI1Nl9maWxlKHBhdGgpKQogICAgUC53cml0ZV9qc29uKG9zLnBhdGguam9pbihT
WydvdXQnXSwgJ2xsbV9pbmZlcmVuY2UuanNvbicpLCBTWydsbG1faW5mbyddKQogICAgUC5fc2F2
ZV9jaGVja3MoUykKICAgIFAubG9nX2V2ZW50KFMsICdMTE0gdHJhaW4gcHJlZGljdGlvbnMgZG9u
ZScsIG1pbnV0ZXM9cm91bmQoaW5mb1snc2Vjb25kcyddIC8gNjAsIDEpLAogICAgICAgICAgICAg
ICAgbGFiZWxfbWFzcz1yb3VuZChpbmZvWydsYWJlbF9tYXNzX21lYW4nXSwgMyksIHNoYTI1Nj1T
WydsbG1faW5mbyddWyd0cmFpbl9sbG1fbnB6X3NoYTI1NiddWzoxNl0pCgoKZGVmIF93cml0ZV9z
dW1tYXJ5KFMpOgogICAgZW52LCBsaSA9IFNbJ2VudiddLCBTWydsbG1faW5mbyddCiAgICByLCB0
ID0gbGlbJ3JlcHJvZHVjaWJpbGl0eSddLCBsaVsndHJhaW4nXQogICAgTCA9IFsnIyBFQ0VSQyBM
aW1pdGF0aW9uIDIsIEV4cGVyaW1lbnQgOGE6IHplcm8tc2hvdCBMTE0gc29mdCBsYWJlbHMgZm9y
IHRoZSB0cmFpbmluZyBzcGxpdCcsICcnLAogICAgICAgICBmJ1J1biB7dGltZS5zdHJmdGltZSgi
JVktJW0tJWQgJUg6JU0iKX0gfCB7ZW52WyJkZXZpY2UiXX0ge2VudlsiZ3B1cyJdfSB8IHRvcmNo
IHtlbnZbInRvcmNoIl19IHwgJwogICAgICAgICBmJ3RyYW5zZm9ybWVycyB7ZW52WyJ0cmFuc2Zv
cm1lcnMiXX0gfCBMTE0ge1NbImNmZyJdWyJsbG1fbW9kZWwiXX0gfCBkYXRhIHNoYTI1NiB7U1si
ZGF0YV9zaGEyNTYiXVs6MTZdfS4uLicsICcnLAogICAgICAgICBmJy0gZGV2IHJlLXNjb3Jpbmcg
KHtyWyJkaWFsb2d1ZXMiXX0gZGlhbG9ndWVzLCB7clsidXR0ZXJhbmNlcyJdfSB1dHRlcmFuY2Vz
KTogJwogICAgICAgICArIChmJ2FyZ21heCBhZ3JlZW1lbnQgd2l0aCBFeHBlcmltZW50IDcgezEw
MCAqIHJbImFyZ21heF9hZ3JlZW1lbnQiXTouMWZ9JSwgbWF4IHxkaWZmfCB7clsibWF4X2Fic19k
aWZmIl06LjRmfScKICAgICAgICAgICAgaWYgJ2FyZ21heF9hZ3JlZW1lbnQnIGluIHIgZWxzZSAn
bm8gcmVmZXJlbmNlJyksCiAgICAgICAgIGYnLSB0cmFpbmluZyBzcGxpdDoge3RbInV0dGVyYW5j
ZXMiXX0gdXR0ZXJhbmNlcywge3RbIm5ld190b2tlbnMiXX0gbmV3IHRva2Vucywge3RbInNlY29u
ZHMiXSAvIDYwOi4wZn0gbWludXRlcywgJwogICAgICAgICBmJ2xhYmVsLXdvcmQgcHJvYmFiaWxp
dHkgbWFzcyB7dFsibGFiZWxfbWFzc19tZWFuIl06LjNmfSBvbiBhdmVyYWdlJywKICAgICAgICAg
ZictIGBkdW1wcy90cmFpbl9sbG0ubnB6YCBTSEEtMjU2OiBge2xpWyJ0cmFpbl9sbG1fbnB6X3No
YTI1NiJdfWAnLAogICAgICAgICAnLSB0aGUgdGVzdCBzcGxpdCB3YXMgbm90IHNjb3JlZCBhbmQg
bm8gdGVzdCBsYWJlbCB3YXMgcmVhZCddCiAgICB3YXJuID0gW2MgZm9yIGMgaW4gU1snY2hlY2tz
J10gaWYgbm90IGNbJ3Bhc3NlZCddXQogICAgTCArPSBbJycsIGYnQ2hlY2tzOiB7c3VtKGNbInBh
c3NlZCJdIGZvciBjIGluIFNbImNoZWNrcyJdKX0gcGFzc2VkLCB7bGVuKHdhcm4pfSB3YXJuaW5n
cycKICAgICAgICAgICsgKCcnIGlmIG5vdCB3YXJuIGVsc2UgJyAtLSAnICsgJzsgJy5qb2luKGYn
e2NbIm5hbWUiXX0gKHtjWyJkZXRhaWwiXX0pJyBmb3IgYyBpbiB3YXJuKSldCiAgICB3aXRoIG9w
ZW4ob3MucGF0aC5qb2luKFNbJ291dCddLCAnU1VNTUFSWV9kaXN0aWxsX3RlYWNoZXIubWQnKSwg
J3cnKSBhcyBmOgogICAgICAgIGYud3JpdGUoJ1xuJy5qb2luKEwpICsgJ1xuJykKCgpkZWYgc3Rh
Z2VfcGFja2FnZShTKToKICAgIF93cml0ZV9zdW1tYXJ5KFMpCiAgICB6cGF0aCA9IG9zLnBhdGgu
am9pbihTWydjZmcnXVsnd29ya19kaXInXSwgJ2VjZXJjX2wyX2Rpc3RpbGxfdGVhY2hlcl9yZXN1
bHRzLnppcCcpCiAgICB3aXRoIHppcGZpbGUuWmlwRmlsZSh6cGF0aCwgJ3cnLCB6aXBmaWxlLlpJ
UF9ERUZMQVRFRCkgYXMgejoKICAgICAgICBmb3Igcm9vdCwgXywgZmlsZXMgaW4gb3Mud2FsayhT
WydvdXQnXSk6CiAgICAgICAgICAgIGZvciBuYW1lIGluIGZpbGVzOgogICAgICAgICAgICAgICAg
ZnVsbCA9IG9zLnBhdGguam9pbihyb290LCBuYW1lKQogICAgICAgICAgICAgICAgei53cml0ZShm
dWxsLCBvcy5wYXRoLnJlbHBhdGgoZnVsbCwgb3MucGF0aC5kaXJuYW1lKFNbJ291dCddKSkpCiAg
ICB3aXRoIG9wZW4ob3MucGF0aC5qb2luKFNbJ291dCddLCAnU1VNTUFSWV9kaXN0aWxsX3RlYWNo
ZXIubWQnKSkgYXMgZjoKICAgICAgICBwcmludChmLnJlYWQoKSkKICAgIHByaW50KGYnUmVzdWx0
cyBhcmNoaXZlOiB7enBhdGh9ICh7b3MucGF0aC5nZXRzaXplKHpwYXRoKSAvIDIgKiogMjA6LjFm
fSBNQiknKQogICAgcmV0dXJuIHpwYXRoCg==
'''),
    'dev_llm.npz': ('...
AAB5Lm5weVBLBQYAAAAAAgACAGkAAADzcAAAAAA=
'''),
}
for name, (digest, blob) in CODE_FILES.items():
    data = base64.b64decode(blob)
    assert hashlib.sha256(data).hexdigest() == digest, f'{name} is corrupted'
    with open(os.path.join(CODE_DIR, name), 'wb') as f:
        f.write(data)
print('Code files written and verified:', ', '.join(CODE_FILES))
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
import l2_posthoc
importlib.reload(l2_posthoc)
import l2_pathb
importlib.reload(l2_pathb)
import l2_expert
importlib.reload(l2_expert)
import l2_zeroshot
importlib.reload(l2_zeroshot)
import l2_distill_teacher as LT
importlib.reload(LT)
S = LT.stage_setup(CONFIG, CODE_DIR)

## 2. MELD features (authors' Google Drive) and data checks (only train and dev labels are read)

In [ ]:
LT.stage_prepare(S)

## 3. The LLM: checks against Experiment 7, dev re-scoring, then the training split (downloads about 19 GB; about 7 hours)

In [ ]:
LT.stage_llm(S)

## 4. Results
Prints the summary and writes `ecerc_l2_distill_teacher_results.zip`. On Kaggle it's under **Output** (`ecerc_l2/ecerc_l2_distill_teacher_results.zip`).

In [ ]:
zpath = LT.stage_package(S)
try:
    from google.colab import files
    files.download(zpath)
except ImportError:
    print('Download this file from the Output panel ->', zpath)